# 09: Nerfstudio Integration for Aria Reconstruction

In this notebook we will create a  reproducible pipeline for extracting isolated 3D meshes from Project Aria recordings. 
We will use the SAM2 semantic masks directly into the pose definitions, forcing the Neural Radiance Field to optimize exclusively on the target object. Finally, we extract a clean, lightweight mesh using a predefined bounding box.

### Pipeline Architecture:
1. **Environment Setup**: Hardware and CUDA validation.
2. **Path Configuration**: Mapping inputs (cropped frames, masks) to dynamic outputs.
3. **Automated Cropping**: Slicing a 15% margin from images and masks to guarantee geometric alignment.
4. **Pose Estimation**: Running `ns-process-data` relying purely on visual features.
5. **Network Training**: Training the `nerfacto` architecture on the masked dataset.
6. **Geometry Extraction**: Exporting the `.ply` pointcloud and `.obj` mesh via Poisson surface reconstruction.

**Note**: We will apply the masks directly to the input images, creating a new dataset with a solid black background. This approach ensures that the NeRF model focuses solely on the object of interest, effectively ignoring any background geometry during training.

As a sample, we will use the frames from the `kettle_and_forklift_recording.vrs` recording, downsampled to 10FPS, located in the local `\data\outputs\segmentation\kettle_and_forklift\sam2\frames_10fps` directory and its corresponding segmentation masks located in `data/outputs/segmentation/kettle_and_forklift/sam2/masks`.

## 9.1 Environment & GPU Validation

Ensure PyTorch is configured correctly for CUDA execution. This step guarantees that Nerfstudio and its underlying dataloaders can access the GPU hardware acceleration required for training.

In [1]:
import torch
import subprocess

# Validate CUDA presence and PyTorch bindings to prevent CPU-fallback during training
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {device_name} ({memory_gb:.2f} GB VRAM)")
else:
    print("WARNING: CUDA is not available. Training will fail or run extremely slow on CPU.")

PyTorch Version: 2.11.0+cu130
CUDA Available: True
Active GPU: NVIDIA GeForce RTX 3060 Laptop GPU (6.00 GB VRAM)


## 9.2 Paths & Processed Dataset Setup

This section defines the directory structure and prepares the processed Nerfstudio dataset generated by the official Aria pipeline. The raw frame splitting is handled by `ns-process-data aria`, so this notebook no longer performs a manual train/eval selection.

In [2]:
import os
import torch
import sys

# Core Project Identifiers
OBJECT_NAME = "kettle_and_forklift"
PROJECT_ROOT = os.path.abspath(os.path.join("..", "data"))

# Raw Inputs (10fps subset and corresponding SAM2 masks)
RAW_FRAMES_10FPS = os.path.join(PROJECT_ROOT, "outputs", "segmentation", OBJECT_NAME, "sam2", "frames_10fps")
RAW_MASKS_DIR = os.path.join(PROJECT_ROOT, "outputs", "segmentation", OBJECT_NAME, "sam2", "masks")

# Target Output Directories for the Pipeline
PIPELINE_ROOT = os.path.join(PROJECT_ROOT, "outputs", "nerfstudio", OBJECT_NAME, "images_10fps_cropped")
IMAGES_CROPPED = os.path.join(PIPELINE_ROOT, "images_cropped")
MASKS = os.path.join(PIPELINE_ROOT, "masks")
TRAINING_DIR = os.path.join(PIPELINE_ROOT, "training")
EXPORT_DIR = os.path.join(PIPELINE_ROOT, "exports")

# Generate directory structure
for directory in [IMAGES_CROPPED, MASKS, TRAINING_DIR, EXPORT_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f"Pipeline root established at: {PIPELINE_ROOT}")

Pipeline root established at: c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\nerfstudio\kettle_and_forklift\images_10fps_cropped


## 9.3 Geometry Alignment
To completely bypass extreme fisheye distortions and allow COLMAP to function natively, we strip the outer 15% of the frame. 

**Note:** This exact crop must be applied simultaneously to both the RGB images and the SAM2 masks to maintain pixel-perfect semantic alignment.

In [ ]:
import os
from PIL import Image
from tqdm import tqdm

CROP_PERCENT = 0.15

def apply_percentage_crop(input_dir, output_dir, extension):
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(extension)]
    print(f"Processing {len(files)} files in {os.path.basename(input_dir)}...")

    for filename in tqdm(files):
        img_path = os.path.join(input_dir, filename)
        with Image.open(img_path) as img:
            w, h = img.size
            
            # Calculate 15% margins
            left = w * CROP_PERCENT
            top = h * CROP_PERCENT
            right = w * (1 - CROP_PERCENT)
            bottom = h * (1 - CROP_PERCENT)
            
            # Apply crop and save
            img_cropped = img.crop((left, top, right, bottom))
            img_cropped.save(os.path.join(output_dir, filename))

# Process both RGB frames and SAM2 masks
apply_percentage_crop(RAW_FRAMES_10FPS, IMAGES_CROPPED, ".jpg")
apply_percentage_crop(RAW_MASKS_DIR, MASKS, ".png")

print(f"\nCropping complete. Files saved to {IMAGES_CROPPED} and {MASKS}")

## 9.4 Pose Estimation (COLMAP)
We can now execute `ns-process-data` using standard pinhole logic. This will generate the base `transforms.json`.

In [40]:
import subprocess

# Standard processing command.
sfm_cmd = [
    "ns-process-data", "images",
    "--data", IMAGES_CROPPED,
    "--output-dir", PIPELINE_ROOT
]

print(f"Executing: {' '.join(sfm_cmd)}")

process = subprocess.Popen(
    sfm_cmd, 
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT, 
    text=True,
    bufsize=1,
    encoding="utf-8",
    errors="replace"
)

# Stream the output in real-time
try:
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
except Exception as e:
    print(f"\nStream error: {e}")

process.wait()

if process.returncode != 0:
    print(f"\n[ERROR] Command failed with exit code {process.returncode}")
else:
    print("\n[SUCCESS] COLMAP processing completed successfully.")

Executing: ns-process-data images --data c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\nerfstudio\kettle_and_forklift\images_10fps_cropped\images_cropped --output-dir C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\nerfstudio\kettle_and_forklift\images_10fps_cropped
( ●    ) Copying images...
(  ●   ) Copying images...
(   ●  ) Copying images...
(    ● ) Copying images...
(     ●) Copying images...
(    ● ) Copying images...
(  ●   ) Copying images...
( ●    ) Copying images...
(●     ) Copying images...
( ●    ) Copying images...
(  ●   ) Copying images...
(   ●  ) Copying images...
(     ●) Copying images...
(    ● ) Copying images...
(   ●  ) Copying images...
(  ●   ) Copying images...
( ●    ) Copying images...
(●     ) Copying images...
( ●    ) Copying images...
(   ●  ) Copying images...
(    ● ) Copying images...
(     ●) Copying images...
(    ● ) Copying images...
(   ●  ) Copying images...
(  ●   ) Copying images...
(●     ) Copying images...

## 9.5 Mask Injection
We will now modify the `transforms.json` to inject the masks, ensuring that each pose points to the corresponding masked image in the new `masks` folder. This step is crucial for guiding the NeRF optimization to focus exclusively on the object of interest.

In [45]:
import json
import os
import re

JSON_PATH = os.path.join(PIPELINE_ROOT, "transforms.json")
if not os.path.exists(JSON_PATH):
    print(f"ERROR: {JSON_PATH} not found. Run ns-process-data before this cell.")
else:
    with open(JSON_PATH, 'r') as f:
        data = json.load(f)

    print(f"Injecting masks for {len(data['frames'])} frames...")

    injection_count = 0
    missing_masks = []

    for frame in data['frames']:
        img_filename = os.path.basename(frame['file_path'])
        
        match = re.search(r'(\d+)', img_filename)
        if match:
            original_number = int(match.group(1))
            mask_number = original_number - 1
            
            mask_filename = f"{mask_number:06d}.png"
            mask_full_path = os.path.join(MASKS, mask_filename)
            
            if os.path.exists(mask_full_path):
                frame["mask_path"] = f"./masks/{mask_filename}"
                injection_count += 1
            else:
                missing_masks.append(mask_filename)

    with open(JSON_PATH, 'w') as f:
        json.dump(data, f, indent=4)

    print(f"\nSUCCESS: {injection_count} masks injected correctly.")
    if missing_masks:
        print(f"WARNING: {len(missing_masks)} masks not found.")
        print(f"Example missing (first 5): {missing_masks[:5]}")

Injecting masks for 505 frames...

SUCCESS: 505 masks injected correctly.


## 9.5 NeRF Training
Launch the `nerfacto` training sequence. Due to the injected masks, the model will exclusively reconstruct the object of interest while ignoring the background geometry.

In [ ]:
import subprocess

train_cmd = [
    "ns-train", "nerfacto",
    "--data", PIPELINE_ROOT,
    "--output-dir", TRAINING_DIR,
    "--pipeline.model.near-plane", "0.05",
    "--pipeline.model.far-plane", "10.0",
    "--pipeline.model.distortion-loss-mult", "0.01"
]


print(f"Executing: {' '.join(train_cmd)}")

# Use subprocess to execute the command and stream output in real-time
process = subprocess.Popen(
    train_cmd, 
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT, 
    text=True,
    bufsize=1,
    encoding="utf-8",
    errors="replace"
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.wait()

if process.returncode != 0:
    print(f"\n[ERROR] Command failed with code {process.returncode}")
else:
    print("\n[SUCCESS] Training has completed.")

Executing: ns-train nerfacto --data c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\nerfstudio\kettle_and_forklift\images_10fps_cropped --output-dir c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\nerfstudio\kettle_and_forklift\images_10fps_cropped\training --pipeline.model.near-plane 0.05 --pipeline.model.far-plane 10.0 --pipeline.model.distortion-loss-mult 0.01
C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\venv\Lib\site-packages\nerfstudio\utils\misc.py:183: RuntimeWarning: Windows does not yet support torch.compile and the performance will be affected.
  warnings.warn(
C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\venv\Lib\site-packages\nerfstudio\field_components\activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\venv\Lib\site-packages\nerfstudio\fie

## 6 Geometry Extraction
Finally, we will extract the geometry using the `ns-export` command, which will output a `.ply` point cloud and a `.obj` mesh. The Poisson surface reconstruction method will be applied to ensure a clean reconstruction.

## 6.1 Export Point Cloud

Export the latest Nerfstudio checkpoint as a point cloud. This uses the same export path as the Nerfstudio viewer and writes the result into the notebook's export directory.

In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

TRAINING_ROOT = Path(TRAINING_DIR)
EXPORT_ROOT = Path(EXPORT_DIR)

def find_latest_config(training_root: Path) -> Path:
    configs = sorted(training_root.rglob('config.yml'), key=lambda path: path.stat().st_mtime, reverse=True)
    if not configs:
        raise FileNotFoundError(f'No config.yml found under {training_root}')
    return configs[0]

latest_config = find_latest_config(TRAINING_ROOT)
pcl_export_dir = EXPORT_ROOT / 'pcl'
pcl_export_dir.mkdir(parents=True, exist_ok=True)

pcl_cmd = [
    'ns-export', 'pointcloud',
    '--load-config', str(latest_config),
    '--output-dir', str(pcl_export_dir),
    '--num-points', '1000000',
    '--remove-outliers', 'True',
    '--normal-method', 'open3d',
    '--save-world-frame', 'True',
]

print(f'Using config: {latest_config}')
print(f"Executing: {' '.join(pcl_cmd)}")

process = subprocess.Popen(
    pcl_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    encoding='utf-8',
    errors='replace',
)

try:
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
except Exception as e:
    print(f'\nStream error: {e}')

process.wait()

if process.returncode != 0:
    print(f'\n[ERROR] Point cloud export failed with code {process.returncode}')
else:
    print(f'\n[SUCCESS] Point cloud export completed in {pcl_export_dir}')

## 6.2 Export Mesh

Export a Poisson mesh from the same latest checkpoint used for the point cloud export. The output is written to the mesh export folder inside the notebook export directory.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

TRAINING_ROOT = Path(TRAINING_DIR)
EXPORT_ROOT = Path(EXPORT_DIR)

def find_latest_config(training_root: Path) -> Path:
    configs = sorted(training_root.rglob('config.yml'), key=lambda path: path.stat().st_mtime, reverse=True)
    if not configs:
        raise FileNotFoundError(f'No config.yml found under {training_root}')
    return configs[0]

latest_config = find_latest_config(TRAINING_ROOT)
mesh_export_dir = EXPORT_ROOT / 'mesh'
mesh_export_dir.mkdir(parents=True, exist_ok=True)

mesh_cmd = [
    'ns-export', 'poisson',
    '--load-config', str(latest_config),
    '--output-dir', str(mesh_export_dir),
    '--target-num-faces', '50000',
    '--num-pixels-per-side', '2048',
    '--num-points', '1000000',
    '--remove-outliers', 'True',
    '--normal-method', 'open3d',
]

print(f'Using config: {latest_config}')
print(f"Executing: {' '.join(mesh_cmd)}")

process = subprocess.Popen(
    mesh_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    encoding='utf-8',
    errors='replace',
)

try:
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
except Exception as e:
    print(f'\nStream error: {e}')

process.wait()

if process.returncode != 0:
    print(f'\n[ERROR] Mesh export failed with code {process.returncode}')
else:
    print(f'\n[SUCCESS] Mesh export completed in {mesh_export_dir}')